[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/course-1-deep-learning/ext-07-embeddings-contrastive-vae.ipynb)

# Advanced Discussion 7 — reading embeddings, contrastive learning, VAE, honest anomaly scores

Reproduces the chapter 7 advanced discussion: PCA versus t-SNE with trustworthiness, SimCLR-style contrastive pre-training with a linear probe, a VAE with its two loss terms, and an autoencoder anomaly detector compared with isolation forest.

**Optional advanced-discussion lab** for Hands-On Deep Learning with Python (AIBits Academy). Not graded and not needed for the certificate. Everything here runs on the Colab free tier (CPU is enough) with built-in datasets, so no download can fail.

Offline fallback: scikit-learn's bundled digits dataset.

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F, numpy as np, time
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE, trustworthiness
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.ensemble import IsolationForest
from sklearn.metrics import roc_auc_score
torch.manual_seed(0); np.random.seed(0)
X, y = load_digits(return_X_y=True); Xn = X / 16.0

## 1. PCA vs t-SNE

In [ ]:
p2 = PCA(2, random_state=0).fit_transform(Xn)
print("PCA-2D trustworthiness(k=10): %.3f" % trustworthiness(Xn, p2, n_neighbors=10))
for perp in (2, 30, 100):
    t0 = time.time()
    e = TSNE(2, perplexity=perp, init="pca", random_state=0).fit(Xn)
    print("t-SNE perplexity %3d: trustworthiness %.3f  KL %.3f  (%.1fs)" % (perp, trustworthiness(Xn, e.embedding_, n_neighbors=10), e.kl_divergence_, time.time() - t0))

## 2. SimCLR-style contrastive learning on 8x8 digits (unlabeled half for pre-training, other half for the probe)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
Xt = torch.tensor(Xn, dtype=torch.float32).view(-1, 1, 8, 8)
tr_i, te_i = train_test_split(np.arange(len(X)), test_size=0.5, random_state=0, stratify=y)
def augment(x, noise=0.10, drop=0.20):
    x = x + noise * torch.randn_like(x)
    x = x * (torch.rand_like(x) > drop).float()
    return x.clamp(0, 1)
def nt_xent(z1, z2, tau=0.5):
    z = F.normalize(torch.cat([z1, z2]), dim=1); n = z1.shape[0]
    sim = z @ z.T / tau
    sim.fill_diagonal_(-1e9)
    targets = torch.cat([torch.arange(n, 2 * n), torch.arange(0, n)])
    return F.cross_entropy(sim, targets)
torch.manual_seed(0); np.random.seed(0)
enc = nn.Sequential(nn.Flatten(), nn.Linear(64, 256), nn.ReLU(), nn.Linear(256, 64))
head = nn.Sequential(nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 32))
opt = torch.optim.Adam(list(enc.parameters()) + list(head.parameters()), lr=2e-3)
for epoch in range(100):
    perm = torch.tensor(np.random.permutation(tr_i))
    for i in range(0, len(perm), 128):
        b = Xt[perm[i:i + 128]]
        if len(b) < 8: continue
        loss = nt_xent(head(enc(augment(b))), head(enc(augment(b))))
        opt.zero_grad(); loss.backward(); opt.step()
def probe(feats, n_labels, reps=10):
    accs = []
    for s in range(reps):
        idx = np.random.RandomState(s).choice(tr_i, n_labels, replace=False)
        clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000)).fit(feats[idx], y[idx])
        accs.append(clf.score(feats[te_i], y[te_i]))
    return np.mean(accs)
with torch.no_grad(): f_ssl = enc(Xt).numpy()
torch.manual_seed(1); rand_enc = nn.Sequential(nn.Flatten(), nn.Linear(64, 256), nn.ReLU(), nn.Linear(256, 64))
with torch.no_grad(): f_rand = rand_enc(Xt).numpy()
for name, f in (("raw pixels", Xn), ("random encoder", f_rand), ("contrastively pre-trained", f_ssl)):
    print("%-26s" % name, " ".join("%d labels: %.3f" % (n, probe(f, n)) for n in (20, 50, 200)))

## 3. VAE

In [ ]:
torch.manual_seed(0); np.random.seed(0)
class VAE(nn.Module):
    def __init__(s, d=4):
        super().__init__()
        s.enc = nn.Sequential(nn.Linear(64, 128), nn.ReLU()); s.mu = nn.Linear(128, d); s.lv = nn.Linear(128, d)
        s.dec = nn.Sequential(nn.Linear(d, 128), nn.ReLU(), nn.Linear(128, 64), nn.Sigmoid())
    def forward(s, x):
        h = s.enc(x); mu, lv = s.mu(h), s.lv(h)
        z = mu + torch.exp(0.5 * lv) * torch.randn_like(mu)            # reparameterization
        return s.dec(z), mu, lv
def vae_loss(xh, x, mu, lv, beta=1.0):
    rec = F.binary_cross_entropy(xh, x, reduction="sum") / x.shape[0]
    kl = -0.5 * torch.sum(1 + lv - mu.pow(2) - lv.exp()) / x.shape[0]
    return rec + beta * kl, rec, kl
vae = VAE(); o = torch.optim.Adam(vae.parameters(), lr=2e-3); Xf = torch.tensor(Xn, dtype=torch.float32)
for ep in range(200):
    perm = torch.randperm(len(Xf))
    for i in range(0, len(Xf), 128):
        b = Xf[perm[i:i + 128]]; xh, mu, lv = vae(b); l, r, k = vae_loss(xh, b, mu, lv)
        o.zero_grad(); l.backward(); o.step()
with torch.no_grad(): xh, mu, lv = vae(Xf); l, r, k = vae_loss(xh, Xf, mu, lv)
print("VAE (d=4): recon %.1f nats, KL %.1f nats per image" % (r.item(), k.item()))
with torch.no_grad(): s = vae.dec(torch.randn(8, 4)); print("sample decoder output shape", tuple(s.shape))

## 4. reconstruction-error anomaly detection: normal = digits 0-8, anomaly = 9

In [ ]:
torch.manual_seed(0); np.random.seed(0)
normal = y != 9
Xnorm = torch.tensor(Xn[normal], dtype=torch.float32)
ae = nn.Sequential(nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 8), nn.ReLU(), nn.Linear(8, 32), nn.ReLU(), nn.Linear(32, 64), nn.Sigmoid())
oa = torch.optim.Adam(ae.parameters(), lr=2e-3)
idx_tr, idx_te = train_test_split(np.arange(len(Xnorm)), test_size=0.3, random_state=0)
for ep in range(300):
    perm = torch.randperm(len(idx_tr))
    for i in range(0, len(perm), 64):
        b = Xnorm[idx_tr][perm[i:i + 64]]; oa.zero_grad(); F.mse_loss(ae(b), b).backward(); oa.step()
with torch.no_grad():
    test_x = torch.cat([Xnorm[idx_te], torch.tensor(Xn[~normal], dtype=torch.float32)])
    lab = np.r_[np.zeros(len(idx_te)), np.ones((~normal).sum())]
    err = ((ae(test_x) - test_x) ** 2).mean(1).numpy()
print("autoencoder reconstruction-error AUROC (9s as anomalies): %.3f" % roc_auc_score(lab, err))
iso = IsolationForest(random_state=0).fit(Xnorm[idx_tr].numpy())
print("isolation forest AUROC: %.3f" % roc_auc_score(lab, -iso.score_samples(test_x.numpy())))
thr = np.quantile(err[lab == 0], 0.95); print("recall of 9s at the 5%% false-alarm threshold: %.3f" % (err[lab == 1] > thr).mean())

## Your turn

Change one setting at a time (seed, learning rate, ε, perplexity, threshold), re-run, and write one sentence about what changed and why. Keep the numbers: your results will differ slightly from the chapter's, and that difference is worth explaining.